# 07 — Khulna Paper-Style ML Downscaling — FINAL ALL OUTPUTS + FULL COVER

This notebook keeps the corrected 00–06D workflow, but restores the **complete visual/output package** of your older `07_Khulna_Paper_Style_PROJ_FREE_FULL_FINAL_v3_ORDERED` notebook.

### It produces
- Input-combination table
- Validation and independent-2022 test metrics
- Paper-style Table 1 as CSV + PNG + PDF
- Spatial station-wise MAE, RMSE and PCC maps
- Observed-vs-predicted plots for every input combination and every available model
- Best model for each input combination
- Overall best model summary
- Monthly 2022 downscaled GeoTIFFs
- Full-Khulna spatial maps using audited nearest-valid filling **inside the Khulna mask**
- Resolution and mask QA
- Monthly map-generation QA
- Annual-generation QA
- Annual rainfall GeoTIFFs
- Final 4 × 3 annual rainfall panel:
  Random Forest / XGBoost / LightGBM / CatBoost × Comb1 / Comb1_land / Comb2_land
- Output manifest

### Important
The modelling table is `station_samples_native_tidy_REPAIRED.csv` from Notebook 06D.

For the spatial maps only, remaining NoData cells **inside the Khulna target mask** are filled from the nearest valid target-grid cell before prediction so the final maps cover the full district, matching the visual behavior of your older notebook. Every filled-cell count is saved in QA. No cells outside Khulna are predicted.


In [ ]:
# ============================================================
# 0. IMPORTS + PROJECT PATHS
# ============================================================

from pathlib import Path
from collections import OrderedDict
import json
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

from scipy.ndimage import distance_transform_edt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import ParameterGrid


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

OUT_ROOT = PROJECT_ROOT / "outputs" / "07_FINAL_ALL_OUTPUTS_FULL_COVER"
TABLE_DIR = OUT_ROOT / "tables"
FIG_DIR = OUT_ROOT / "figures"
MAP_DIR = OUT_ROOT / "monthly_maps"
ANNUAL_DIR = OUT_ROOT / "annual_maps"
QA_DIR = OUT_ROOT / "quality_control"
MODEL_DIR = PROJECT_ROOT / "models" / "07_FINAL_ALL_OUTPUTS_FULL_COVER"

for folder in [
    OUT_ROOT,
    TABLE_DIR,
    FIG_DIR,
    MAP_DIR,
    ANNUAL_DIR,
    QA_DIR,
    MODEL_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

FIG_DPI = 350
RANDOM_STATE = 42

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUT_ROOT     =", OUT_ROOT)


In [ ]:
# ============================================================
# 1. LOAD CORRECTED 06D MODELLING TABLE
# ============================================================

SAMPLES_PATH = (
    PROCESSED_DIR
    / "station_samples_native_tidy_REPAIRED.csv"
)

if not SAMPLES_PATH.exists():
    raise FileNotFoundError(
        "station_samples_native_tidy_REPAIRED.csv not found. "
        "Run 06D_Local_Pixel_Hole_Repair_FIXED.ipynb first."
    )

data = pd.read_csv(SAMPLES_PATH)

TARGET = "rainfall_mm"

TRAIN_YEARS = [2017, 2018, 2019, 2020]
VALID_YEARS = [2021]
TEST_YEARS = [2022]
TEST_YEAR = 2022

PRECIP8 = [
    "CCS",
    "PDIR",
    "GSMaP_MVK",
    "CDR",
    "CHIRPS",
    "IMERG",
    "GSMaP_Gauge",
    "ERA5",
]

LAND = [
    "DEM",
    "NDVI",
    "LST_Day",
    "Distance_Sea",
]

required = [
    "station_id",
    "year",
    "month",
    "latitude",
    "longitude",
    TARGET,
] + PRECIP8 + LAND

missing_cols = [
    c for c in required
    if c not in data.columns
]

if missing_cols:
    raise KeyError(
        f"Required columns are missing: {missing_cols}"
    )

print("Data shape:", data.shape)
print("Stations:", data["station_id"].nunique())
print("Years:", sorted(data["year"].unique()))
display(data.head())


In [ ]:
# ============================================================
# 2. PAPER-STYLE INPUT COMBINATIONS
# ============================================================

COMBINATIONS = OrderedDict()

COMBINATIONS["Comb1"] = PRECIP8
COMBINATIONS["Comb1_land"] = PRECIP8 + LAND

# Paper-style reduced precipitation set
COMBINATIONS["Comb2"] = [
    "CHIRPS",
    "CDR",
    "ERA5",
]

COMBINATIONS["Comb2_land"] = [
    "CHIRPS",
    "CDR",
    "ERA5",
] + LAND

# Optional CCS/PDIR experiment, retained from the older notebook style
COMBINATIONS["Comb3"] = [
    "CCS",
    "PDIR",
]

COMBINATIONS["Comb3_land"] = [
    "CCS",
    "PDIR",
] + LAND

# Single precipitation product + land combinations
for p in PRECIP8:
    COMBINATIONS[f"{p}_land"] = [p] + LAND

for k in list(COMBINATIONS):
    COMBINATIONS[k] = list(
        dict.fromkeys(
            COMBINATIONS[k]
        )
    )

combo_table = pd.DataFrame({
    "combination": list(COMBINATIONS.keys()),
    "features": [
        ", ".join(COMBINATIONS[k])
        for k in COMBINATIONS
    ],
})

display(combo_table)

combo_table.to_csv(
    TABLE_DIR / "input_combinations.csv",
    index=False
)


In [ ]:
# ============================================================
# 3. MODEL AVAILABILITY + HYPERPARAMETER GRIDS
# ============================================================

MODELS = OrderedDict()

# Optional Cubist
try:
    from cubist import Cubist

    MODELS["Cubist"] = (
        Cubist,
        {
            "n_committees": [5, 10],
            "neighbors": [0, 3],
        }
    )

except Exception as e:
    print("Cubist unavailable:", e)


MODELS["Random Forest"] = (
    RandomForestRegressor,
    {
        "n_estimators": [300],
        "max_depth": [None, 15],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", 1.0],
    }
)


try:
    from xgboost import XGBRegressor

    MODELS["XGBoost"] = (
        XGBRegressor,
        {
            "n_estimators": [300],
            "max_depth": [3, 5],
            "learning_rate": [0.03, 0.08],
            "subsample": [0.8],
            "colsample_bytree": [0.9],
        }
    )

except Exception as e:
    print("XGBoost unavailable:", e)


try:
    from lightgbm import LGBMRegressor

    MODELS["LightGBM"] = (
        LGBMRegressor,
        {
            "n_estimators": [300],
            "num_leaves": [15, 31],
            "learning_rate": [0.03, 0.08],
        }
    )

except Exception as e:
    print("LightGBM unavailable:", e)


try:
    from catboost import CatBoostRegressor

    MODELS["CatBoost"] = (
        CatBoostRegressor,
        {
            "iterations": [300],
            "depth": [5, 7],
            "learning_rate": [0.03, 0.08],
        }
    )

except Exception as e:
    print("CatBoost unavailable:", e)


print("Available models:", list(MODELS.keys()))


In [ ]:
# ============================================================
# 4. MODEL HELPERS
# ============================================================

def pcc(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)

    m = np.isfinite(y) & np.isfinite(p)

    if (
        m.sum() < 2
        or np.std(y[m]) == 0
        or np.std(p[m]) == 0
    ):
        return np.nan

    return float(
        np.corrcoef(
            y[m],
            p[m]
        )[0, 1]
    )


def metric_dict(y, p):
    return {
        "MAE": float(
            mean_absolute_error(
                y,
                p
            )
        ),
        "RMSE": float(
            np.sqrt(
                mean_squared_error(
                    y,
                    p
                )
            )
        ),
        "PCC": pcc(
            y,
            p
        ),
        "R2": float(
            r2_score(
                y,
                p
            )
        ),
    }


def make_model(name, cls, params):
    q = dict(params)

    if name == "Random Forest":
        q.update(
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    elif name == "XGBoost":
        q.update(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            objective="reg:squarederror",
            eval_metric="rmse"
        )

    elif name == "LightGBM":
        q.update(
            random_state=RANDOM_STATE,
            verbosity=-1
        )

    elif name == "CatBoost":
        q.update(
            random_seed=RANDOM_STATE,
            verbose=False,
            loss_function="RMSE"
        )

    return cls(
        **q
    )


def fit_combo(
    features,
    model_name,
    cls,
    param_grid
):
    keep = [
        "station_id",
        "year",
        "month",
        "latitude",
        "longitude",
        TARGET,
    ] + features

    d = (
        data[keep]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna(
            subset=[TARGET] + features
        )
        .copy()
    )

    tr = d["year"].isin(
        TRAIN_YEARS
    )

    va = d["year"].isin(
        VALID_YEARS
    )

    te = d["year"].isin(
        TEST_YEARS
    )

    if tr.sum() < 30:
        raise ValueError(
            f"Too few training rows: {tr.sum()}"
        )

    if va.sum() == 0:
        raise ValueError(
            "No validation rows."
        )

    if te.sum() == 0:
        raise ValueError(
            "No 2022 test rows."
        )

    Xtr = d.loc[
        tr,
        features
    ].to_numpy(
        dtype=float
    )

    ytr = d.loc[
        tr,
        TARGET
    ].to_numpy(
        dtype=float
    )

    Xva = d.loc[
        va,
        features
    ].to_numpy(
        dtype=float
    )

    yva = d.loc[
        va,
        TARGET
    ].to_numpy(
        dtype=float
    )

    # Hyperparameter selection uses VALIDATION ONLY.
    best = None

    for params in ParameterGrid(
        param_grid
    ):
        est = make_model(
            model_name,
            cls,
            params
        )

        est.fit(
            Xtr,
            ytr
        )

        pv = np.maximum(
            est.predict(
                Xva
            ),
            0
        )

        vm = metric_dict(
            yva,
            pv
        )

        if (
            best is None
            or vm["RMSE"] < best["valid_metrics"]["RMSE"]
        ):
            best = {
                "params": params,
                "valid_metrics": vm,
            }

    # Refit with Train + Validation.
    dev = tr | va

    est = make_model(
        model_name,
        cls,
        best["params"]
    )

    est.fit(
        d.loc[
            dev,
            features
        ].to_numpy(
            dtype=float
        ),
        d.loc[
            dev,
            TARGET
        ].to_numpy(
            dtype=float
        )
    )

    yte = d.loc[
        te,
        TARGET
    ].to_numpy(
        dtype=float
    )

    pte = np.maximum(
        est.predict(
            d.loc[
                te,
                features
            ].to_numpy(
                dtype=float
            )
        ),
        0
    )

    return {
        "features": features,
        "estimator": est,
        "params": best["params"],
        "valid_metrics": best["valid_metrics"],
        "test_frame": d.loc[
            te
        ].copy(),
        "y_test": yte,
        "pred_test": pte,
        "N_train": int(
            tr.sum()
        ),
        "N_valid": int(
            va.sum()
        ),
        "N_test": int(
            te.sum()
        ),
        "N_train_valid": int(
            dev.sum()
        ),
    }


In [ ]:
# ============================================================
# 5. TRAIN / VALIDATE / INDEPENDENT 2022 TEST
# ============================================================

trained = {}
rows = []

for combo, features in COMBINATIONS.items():

    print(
        "\n==",
        combo,
        "=="
    )

    for model_name, (
        cls,
        grid
    ) in MODELS.items():

        try:
            bundle = fit_combo(
                features,
                model_name,
                cls,
                grid
            )

            trained[
                (combo, model_name)
            ] = bundle

            mm = metric_dict(
                bundle["y_test"],
                bundle["pred_test"]
            )

            rows.append({
                "Input": combo,
                "Model": model_name,
                "Validation_MAE": bundle["valid_metrics"]["MAE"],
                "Validation_RMSE": bundle["valid_metrics"]["RMSE"],
                "Validation_PCC": bundle["valid_metrics"]["PCC"],
                "Validation_R2": bundle["valid_metrics"]["R2"],
                "MAE": mm["MAE"],
                "RMSE": mm["RMSE"],
                "PCC": mm["PCC"],
                "R2": mm["R2"],
                "N_train": bundle["N_train"],
                "N_valid": bundle["N_valid"],
                "N_train_valid": bundle["N_train_valid"],
                "N_test": bundle["N_test"],
                "BestParams": json.dumps(
                    bundle["params"],
                    default=str
                ),
            })

            joblib.dump(
                bundle,
                MODEL_DIR
                / f"{combo}__{model_name.replace(' ', '_')}.joblib"
            )

            print(
                model_name,
                mm,
                "N_test=",
                bundle["N_test"]
            )

        except Exception as e:
            print(
                "SKIP",
                model_name,
                repr(e)
            )


results_long = pd.DataFrame(
    rows
)

if results_long.empty:
    raise RuntimeError(
        "No model completed."
    )

results_long.to_csv(
    TABLE_DIR
    / "table1_metrics_long.csv",
    index=False
)

display(
    results_long
    .sort_values(
        ["RMSE", "MAE"]
    )
    .reset_index(
        drop=True
    )
)


In [ ]:
# ============================================================
# 6. PAPER-STYLE TABLE 1 — CSV + PNG + PDF
# ============================================================

model_order = [
    m for m in [
        "Cubist",
        "Random Forest",
        "XGBoost",
        "LightGBM",
        "CatBoost",
    ]
    if m in results_long["Model"].unique()
]

input_order = [
    x for x in COMBINATIONS
    if x in results_long["Input"].unique()
]

rows = []

for i, inp in enumerate(
    input_order,
    1
):
    rec = {
        "No": i,
        "Input": inp,
    }

    for model in model_order:
        r = results_long[
            (results_long["Input"] == inp)
            &
            (results_long["Model"] == model)
        ]

        for metric in [
            "MAE",
            "RMSE",
            "PCC",
        ]:
            rec[
                f"{model} | {metric}"
            ] = (
                float(
                    r.iloc[0][metric]
                )
                if len(r)
                else np.nan
            )

    rows.append(
        rec
    )


table1 = pd.DataFrame(
    rows
)

table1.to_csv(
    TABLE_DIR
    / "Table_1_paper_style.csv",
    index=False
)


fig, ax = plt.subplots(
    figsize=(
        max(
            12,
            1.2 * len(
                table1.columns
            )
        ),
        max(
            5,
            0.34 * (
                len(table1) + 3
            )
        )
    )
)

ax.axis(
    "off"
)

fmt = table1.copy()

for c in fmt.columns:

    if c not in [
        "No",
        "Input"
    ]:

        fmt[c] = fmt[c].map(
            lambda x:
                ""
                if pd.isna(x)
                else (
                    f"{x:.3f}"
                    if c.endswith(
                        "PCC"
                    )
                    else f"{x:.2f}"
                )
        )


tbl = ax.table(
    cellText=fmt.values,
    colLabels=fmt.columns,
    cellLoc="center",
    loc="center"
)

tbl.auto_set_font_size(
    False
)

tbl.set_fontsize(
    6
)

tbl.scale(
    1,
    1.25
)

ax.set_title(
    "Table 1. 2022 independent-test performance",
    fontweight="bold",
    pad=12
)

fig.tight_layout()

fig.savefig(
    TABLE_DIR
    / "Table_1_paper_style.png",
    dpi=FIG_DPI,
    bbox_inches="tight"
)

fig.savefig(
    TABLE_DIR
    / "Table_1_paper_style.pdf",
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# 7. BEST MODEL FOR EACH INPUT + OVERALL BEST
#    Selection is based on VALIDATION RMSE.
# ============================================================

best_each_input = (
    results_long
    .sort_values(
        [
            "Input",
            "Validation_RMSE",
            "Validation_MAE",
        ]
    )
    .groupby(
        "Input",
        as_index=False
    )
    .first()
)

print(
    "\nBEST MODEL FOR EACH INPUT COMBINATION"
)

display(
    best_each_input[
        [
            "Input",
            "Model",
            "Validation_MAE",
            "Validation_RMSE",
            "Validation_PCC",
            "MAE",
            "RMSE",
            "PCC",
            "N_test",
        ]
    ]
)

best_each_input.to_csv(
    TABLE_DIR
    / "Best_Model_for_Each_Input.csv",
    index=False
)


overall_best = (
    best_each_input
    .sort_values(
        [
            "Validation_RMSE",
            "Validation_MAE",
        ]
    )
    .iloc[0]
)

print("\n" + "=" * 70)
print("OVERALL BEST MODEL BY VALIDATION RMSE")
print("=" * 70)
print(
    "Input combination :",
    overall_best["Input"]
)
print(
    "Model             :",
    overall_best["Model"]
)
print(
    f"Validation RMSE   : {overall_best['Validation_RMSE']:.2f} mm"
)
print(
    f"2022 Test RMSE    : {overall_best['RMSE']:.2f} mm"
)
print(
    f"2022 Test MAE     : {overall_best['MAE']:.2f} mm"
)
print(
    f"2022 Test PCC     : {overall_best['PCC']:.3f}"
)
print("=" * 70)


In [ ]:
# ============================================================
# 8. LOAD KHULNA BOUNDARY FOR VISUAL STATION MAPS
# ============================================================

import geopandas as gpd

boundary_dir = DATA_DIR / "raw" / "boundary"

boundary_candidates = [
    *boundary_dir.rglob("*.shp"),
    *boundary_dir.rglob("*.gpkg"),
    *boundary_dir.rglob("*.geojson"),
]

if not boundary_candidates:
    raise FileNotFoundError(
        "Khulna boundary not found."
    )

boundary_path = boundary_candidates[0]

boundary = gpd.read_file(
    boundary_path
)

WGS84_PROJ = (
    "+proj=longlat +datum=WGS84 +no_defs"
)

try:
    boundary = boundary.to_crs(
        WGS84_PROJ
    )
except Exception as e:
    print(
        "Boundary CRS transform skipped:",
        e
    )

print(
    "Boundary:",
    boundary_path
)


In [ ]:
# ============================================================
# 9. SPATIAL STATION MAE / RMSE / PCC MAPS
# ============================================================

SPATIAL_COMBO = next(
    (
        c for c in [
            "Comb1_land",
            "Comb2_land",
            "Comb1",
        ]
        if c in COMBINATIONS
    ),
    None
)


def station_metric_table(
    combo,
    model
):
    b = trained[
        (combo, model)
    ]

    d = b[
        "test_frame"
    ].copy()

    d[
        "prediction"
    ] = b[
        "pred_test"
    ]

    rr = []

    for sid, g in d.groupby(
        "station_id"
    ):
        mm = metric_dict(
            g[TARGET],
            g["prediction"]
        )

        rr.append({
            "station": sid,
            "latitude": g["latitude"].iloc[0],
            "longitude": g["longitude"].iloc[0],
            **mm,
        })

    return pd.DataFrame(
        rr
    )


station_tables = {}

for model in model_order:

    if (
        SPATIAL_COMBO,
        model
    ) in trained:

        station_tables[
            model
        ] = station_metric_table(
            SPATIAL_COMBO,
            model
        )

        station_tables[
            model
        ].to_csv(
            TABLE_DIR
            / (
                f"station_metrics_{SPATIAL_COMBO}_"
                f"{model.replace(' ', '_')}.csv"
            ),
            index=False
        )


def plot_station_metric(
    metric,
    stem,
    cmap
):
    names = list(
        station_tables
    )

    vals = np.concatenate([
        station_tables[m][metric]
        .dropna()
        .values
        for m in names
    ])

    vmin, vmax = np.nanpercentile(
        vals,
        [2, 98]
    )

    fig, axes = plt.subplots(
        1,
        len(names),
        figsize=(
            2.8 * len(names),
            4.4
        ),
        squeeze=False
    )

    sc = None

    for j, m in enumerate(
        names
    ):
        ax = axes[
            0,
            j
        ]

        boundary.boundary.plot(
            ax=ax,
            color="black",
            linewidth=0.7
        )

        t = station_tables[
            m
        ]

        sc = ax.scatter(
            t["longitude"],
            t["latitude"],
            c=t[metric],
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            s=45,
            edgecolor="black",
            linewidth=0.25
        )

        ax.set_title(
            m
        )

        ax.set_aspect(
            "equal"
        )

        ax.set_xlabel(
            "Longitude"
        )

        if j == 0:
            ax.set_ylabel(
                "Latitude"
            )

    cb = fig.colorbar(
        sc,
        ax=axes.ravel().tolist(),
        shrink=0.72,
        pad=0.02
    )

    cb.set_label(
        metric
        + (
            " (mm)"
            if metric in [
                "MAE",
                "RMSE"
            ]
            else ""
        )
    )

    fig.suptitle(
        f"Spatial distribution of {metric} — {SPATIAL_COMBO}"
    )

    fig.savefig(
        FIG_DIR
        / f"{stem}.png",
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    fig.savefig(
        FIG_DIR
        / f"{stem}.pdf",
        bbox_inches="tight"
    )

    plt.show()


plot_station_metric(
    "MAE",
    "Fig_3_spatial_MAE",
    "YlOrRd"
)

plot_station_metric(
    "RMSE",
    "Fig_4_spatial_RMSE",
    "YlOrRd"
)

plot_station_metric(
    "PCC",
    "Fig_5_spatial_PCC",
    "RdYlGn"
)


In [ ]:
# ============================================================
# 10. OBSERVED vs PREDICTED — ALL COMBINATIONS × ALL MODELS
# ============================================================

MODEL_ORDER = [
    "Cubist",
    "Random Forest",
    "XGBoost",
    "LightGBM",
    "CatBoost",
]

ALL_COMBOS = [
    combo
    for combo in COMBINATIONS.keys()
    if any(
        (
            combo,
            model
        ) in trained
        for model in MODEL_ORDER
    )
]

print(
    "Total combinations:",
    len(ALL_COMBOS)
)

print(
    ALL_COMBOS
)


def plot_all_models_for_combo(
    combo
):

    models = [
        model
        for model in MODEL_ORDER
        if (
            combo,
            model
        ) in trained
    ]

    if len(
        models
    ) == 0:
        return None

    values = []

    for model in models:

        b = trained[
            (
                combo,
                model
            )
        ]

        obs = np.asarray(
            b["y_test"],
            dtype=float
        )

        pred = np.asarray(
            b["pred_test"],
            dtype=float
        )

        values.extend(
            obs[
                np.isfinite(
                    obs
                )
            ]
        )

        values.extend(
            pred[
                np.isfinite(
                    pred
                )
            ]
        )

    values = np.asarray(
        values
    )

    lo = max(
        0,
        np.nanmin(
            values
        )
    )

    hi = np.nanmax(
        values
    )

    pad = (
        (
            hi - lo
        ) * 0.05
        if hi > lo
        else 1
    )

    plot_lo = max(
        0,
        lo - pad
    )

    plot_hi = hi + pad


    if len(models) == 5:

        fig = plt.figure(
            figsize=(
                12,
                7.5
            )
        )

        positions = [
            [0.06, 0.55, 0.26, 0.34],
            [0.37, 0.55, 0.26, 0.34],
            [0.68, 0.55, 0.26, 0.34],
            [0.21, 0.08, 0.26, 0.34],
            [0.53, 0.08, 0.26, 0.34],
        ]

        axes = [
            fig.add_axes(
                pos
            )
            for pos in positions
        ]

    else:

        ncols = min(
            3,
            len(models)
        )

        nrows = int(
            np.ceil(
                len(models)
                / ncols
            )
        )

        fig, axs = plt.subplots(
            nrows,
            ncols,
            figsize=(
                4 * ncols,
                4 * nrows
            )
        )

        axes = np.atleast_1d(
            axs
        ).ravel()

        for ax in axes[
            len(models):
        ]:
            ax.axis(
                "off"
            )


    rows = []

    for ax, model in zip(
        axes,
        models
    ):

        b = trained[
            (
                combo,
                model
            )
        ]

        y_obs = np.asarray(
            b["y_test"],
            dtype=float
        )

        y_pred = np.asarray(
            b["pred_test"],
            dtype=float
        )

        valid = (
            np.isfinite(
                y_obs
            )
            &
            np.isfinite(
                y_pred
            )
        )

        y_obs = y_obs[
            valid
        ]

        y_pred = y_pred[
            valid
        ]

        mae = mean_absolute_error(
            y_obs,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_obs,
                y_pred
            )
        )

        pcc_value = pcc(
            y_obs,
            y_pred
        )

        rows.append({
            "Input": combo,
            "Model": model,
            "MAE": mae,
            "RMSE": rmse,
            "PCC": pcc_value,
            "N_test": len(
                y_obs
            ),
        })

        ax.scatter(
            y_obs,
            y_pred,
            s=28,
            alpha=0.65,
            edgecolors="black",
            linewidths=0.25
        )

        ax.plot(
            [
                plot_lo,
                plot_hi
            ],
            [
                plot_lo,
                plot_hi
            ],
            "--",
            color="black",
            linewidth=1
        )

        ax.set_xlim(
            plot_lo,
            plot_hi
        )

        ax.set_ylim(
            plot_lo,
            plot_hi
        )

        ax.set_aspect(
            "equal",
            adjustable="box"
        )

        ax.set_xlabel(
            "Observed rainfall (mm)"
        )

        ax.set_ylabel(
            "Predicted rainfall (mm)"
        )

        ax.set_title(
            model,
            fontweight="bold"
        )

        txt = (
            f"MAE = {mae:.2f} mm\n"
            f"RMSE = {rmse:.2f} mm\n"
            f"PCC = {pcc_value:.3f}\n"
            f"N = {len(y_obs)}"
        )

        ax.text(
            0.04,
            0.96,
            txt,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                alpha=0.80
            )
        )

        ax.grid(
            alpha=0.20,
            linewidth=0.5
        )


    fig.suptitle(
        f"Observed vs Predicted Rainfall — {combo}",
        fontsize=14,
        fontweight="bold",
        y=0.98
    )

    png_path = (
        FIG_DIR
        / (
            f"Observed_vs_Predicted_"
            f"{combo}_ALL_MODELS.png"
        )
    )

    pdf_path = (
        FIG_DIR
        / (
            f"Observed_vs_Predicted_"
            f"{combo}_ALL_MODELS.pdf"
        )
    )

    fig.savefig(
        png_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight"
    )

    plt.show()

    return pd.DataFrame(
        rows
    )


all_comparison_tables = []

for combo in ALL_COMBOS:

    print(
        "\n" + "=" * 70
    )

    print(
        "PLOTTING:",
        combo
    )

    print(
        "=" * 70
    )

    result = plot_all_models_for_combo(
        combo
    )

    if result is not None:
        all_comparison_tables.append(
            result
        )


all_model_comparison = pd.concat(
    all_comparison_tables,
    ignore_index=True
)

all_model_comparison[
    "Rank_within_Input"
] = (
    all_model_comparison
    .groupby(
        "Input"
    )[
        "RMSE"
    ]
    .rank(
        method="min",
        ascending=True
    )
    .astype(
        int
    )
)

all_model_comparison = (
    all_model_comparison
    .sort_values(
        [
            "Input",
            "Rank_within_Input",
            "RMSE"
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    all_model_comparison
)

all_model_comparison.to_csv(
    TABLE_DIR
    / "Observed_vs_Predicted_ALL_COMBINATIONS_ALL_MODELS.csv",
    index=False
)


In [ ]:
# ============================================================
# 11. TARGET GRID + FULL-COVER SPATIAL FILLING
# ============================================================

GRID_ROOT = (
    PROCESSED_DIR
    / "test2022_target_grid"
)

if not GRID_ROOT.exists():
    raise FileNotFoundError(
        "test2022_target_grid not found. Run Notebook 05 first."
    )


def grid_path(
    feature,
    month
):
    if feature in [
        "DEM",
        "Distance_Sea"
    ]:
        return (
            GRID_ROOT
            / "static"
            / f"{feature}.tif"
        )

    return (
        GRID_ROOT
        / feature
        / f"{feature}_2022_{month:02d}.tif"
    )


def read_grid(
    path
):
    with rasterio.open(
        path
    ) as src:

        arr = src.read(
            1
        ).astype(
            "float32"
        )

        nodata = src.nodata

        if nodata is not None:
            arr[
                np.isclose(
                    arr,
                    nodata
                )
            ] = np.nan

        arr[
            ~np.isfinite(
                arr
            )
        ] = np.nan

        profile = src.profile.copy()
        bounds = src.bounds

    return (
        arr,
        profile,
        bounds
    )


# DEM from Notebook 05 already carries the Khulna mask.
DEM_GRID = (
    GRID_ROOT
    / "static"
    / "DEM.tif"
)

dem_arr, TARGET_PROFILE, TARGET_BOUNDS = read_grid(
    DEM_GRID
)

TARGET_MASK = np.isfinite(
    dem_arr
)

TARGET_SHAPE = dem_arr.shape

TARGET_HEIGHT = TARGET_SHAPE[0]
TARGET_WIDTH = TARGET_SHAPE[1]

TARGET_PIXELS = int(
    TARGET_MASK.sum()
)

NODATA = -9999.0

print(
    "Target shape:",
    TARGET_SHAPE
)

print(
    "Khulna pixels:",
    TARGET_PIXELS
)


def fill_inside_khulna_nearest(
    arr,
    mask
):
    """
    Fill only missing cells INSIDE the Khulna mask.
    Outside Khulna always remains NaN.

    Returns:
      filled_array,
      number_of_cells_filled
    """

    a = np.asarray(
        arr,
        dtype="float32"
    ).copy()

    a[
        ~mask
    ] = np.nan

    valid = (
        np.isfinite(
            a
        )
        & mask
    )

    missing_inside = (
        mask
        & ~valid
    )

    n_missing = int(
        missing_inside.sum()
    )

    if n_missing == 0:
        return (
            a,
            0
        )

    if not valid.any():
        raise ValueError(
            "No valid source values exist inside Khulna."
        )

    # distance_transform_edt returns indices of nearest zero.
    # Use invalid=1 and valid=0 so missing cells point to nearest valid cell.
    invalid = ~valid

    indices = distance_transform_edt(
        invalid,
        return_distances=False,
        return_indices=True
    )

    nearest = a[
        tuple(
            indices
        )
    ]

    a[
        missing_inside
    ] = nearest[
        missing_inside
    ]

    a[
        ~mask
    ] = np.nan

    return (
        a,
        n_missing
    )


def write_masked_tif(
    path,
    arr
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    out = np.where(
        np.isfinite(
            arr
        )
        & TARGET_MASK,
        arr,
        NODATA
    ).astype(
        "float32"
    )

    profile = TARGET_PROFILE.copy()

    profile.update(
        driver="GTiff",
        height=TARGET_HEIGHT,
        width=TARGET_WIDTH,
        count=1,
        dtype="float32",
        nodata=NODATA,
        compress="deflate",
        predictor=3
    )

    with rasterio.open(
        path,
        "w",
        **profile
    ) as dst:
        dst.write(
            out,
            1
        )


In [ ]:
# ============================================================
# 12. GENERATE ALL 2022 MONTHLY MAPS
#     4 MAIN MODELS × 3 MAIN COMBINATIONS
# ============================================================

MAP_COMBOS = [
    c for c in [
        "Comb1",
        "Comb1_land",
        "Comb2_land",
    ]
    if c in COMBINATIONS
]

MAP_MODELS = [
    m for m in [
        "Random Forest",
        "XGBoost",
        "LightGBM",
        "CatBoost",
    ]
    if m in model_order
]


def predict_month_full_cover(
    bundle,
    month
):
    arrays = []

    fill_audit = []

    for feature in bundle[
        "features"
    ]:

        path = grid_path(
            feature,
            month
        )

        if not path.exists():
            return (
                None,
                f"missing file: {path}",
                []
            )

        arr, _, _ = read_grid(
            path
        )

        if arr.shape != TARGET_SHAPE:
            return (
                None,
                f"shape mismatch: {path}",
                []
            )

        # Full-cover behavior matching the old notebook:
        # fill remaining target-grid gaps only INSIDE Khulna.
        filled, n_filled = fill_inside_khulna_nearest(
            arr,
            TARGET_MASK
        )

        if feature in PRECIP8:
            filled[
                filled < 0
            ] = 0

        arrays.append(
            filled
        )

        fill_audit.append({
            "feature": feature,
            "filled_pixels_inside_khulna": n_filled,
        })

    X = np.column_stack([
        a[
            TARGET_MASK
        ]
        for a in arrays
    ])

    if not np.all(
        np.isfinite(
            X
        )
    ):
        return (
            None,
            "non-finite values remain after full-cover fill",
            fill_audit
        )

    pred = np.full(
        TARGET_SHAPE,
        np.nan,
        dtype="float32"
    )

    pred_vals = np.maximum(
        bundle[
            "estimator"
        ].predict(
            X
        ),
        0
    ).astype(
        "float32"
    )

    pred[
        TARGET_MASK
    ] = pred_vals

    pred[
        ~TARGET_MASK
    ] = np.nan

    return (
        pred,
        "OK",
        fill_audit
    )


generated = {}
map_qa = []
fill_qa = []


for combo in MAP_COMBOS:

    for model in MAP_MODELS:

        if (
            combo,
            model
        ) not in trained:
            continue

        bundle = trained[
            (
                combo,
                model
            )
        ]

        out_dir = (
            MAP_DIR
            / combo
            / model.replace(
                " ",
                "_"
            )
        )

        out_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        print(
            "Predicting:",
            combo,
            "|",
            model
        )

        for month in range(
            1,
            13
        ):

            out_tif = (
                out_dir
                / f"downscaled_{TEST_YEAR}_{month:02d}.tif"
            )

            try:
                pred, status, audit = predict_month_full_cover(
                    bundle,
                    month
                )

                if pred is None:
                    map_qa.append({
                        "combo": combo,
                        "model": model,
                        "month": month,
                        "status": "SKIP",
                        "reason": status,
                        "coverage_pct_inside_khulna": 0.0,
                    })

                    print(
                        "  skip",
                        month,
                        status
                    )

                    continue

                write_masked_tif(
                    out_tif,
                    pred
                )

                generated[
                    (
                        combo,
                        model,
                        month
                    )
                ] = out_tif

                coverage = (
                    100.0
                    * (
                        np.isfinite(
                            pred
                        )
                        & TARGET_MASK
                    ).sum()
                    / TARGET_PIXELS
                )

                map_qa.append({
                    "combo": combo,
                    "model": model,
                    "month": month,
                    "status": "OK",
                    "reason": "",
                    "coverage_pct_inside_khulna": coverage,
                })

                for rec in audit:
                    fill_qa.append({
                        "combo": combo,
                        "model": model,
                        "month": month,
                        **rec,
                    })

                print(
                    f"  month {month:02d}: "
                    f"{coverage:.2f}% coverage"
                )

            except Exception as e:
                map_qa.append({
                    "combo": combo,
                    "model": model,
                    "month": month,
                    "status": "ERROR",
                    "reason": repr(e),
                    "coverage_pct_inside_khulna": 0.0,
                })

                print(
                    "  ERROR",
                    month,
                    repr(e)
                )


map_qa = pd.DataFrame(
    map_qa
)

fill_qa = pd.DataFrame(
    fill_qa
)

map_qa.to_csv(
    QA_DIR
    / "map_generation_QA.csv",
    index=False
)

fill_qa.to_csv(
    QA_DIR
    / "full_cover_fill_QA.csv",
    index=False
)

print("\nMAP GENERATION QA")
display(map_qa)

print("\nFULL-COVER FILL QA")
display(fill_qa)

summary = (
    map_qa
    .groupby(
        [
            "combo",
            "model"
        ]
    )
    .agg(
        months_generated=(
            "status",
            lambda x:
                int(
                    x.eq(
                        "OK"
                    ).sum()
                )
        ),
        mean_coverage_pct=(
            "coverage_pct_inside_khulna",
            "mean"
        ),
        min_coverage_pct=(
            "coverage_pct_inside_khulna",
            "min"
        ),
    )
    .reset_index()
)

display(
    summary
)

summary.to_csv(
    QA_DIR
    / "map_generation_summary.csv",
    index=False
)


In [ ]:
# ============================================================
# 13. RESOLUTION + MASK QA
# ============================================================

qa_rows = []

for feature in (
    PRECIP8
    + [
        "NDVI",
        "LST_Day",
        "DEM",
        "Distance_Sea",
    ]
):

    p = grid_path(
        feature,
        1
    )

    if not p.exists():
        continue

    with rasterio.open(
        p
    ) as src:

        qa_rows.append({
            "dataset": feature,
            "xres": abs(
                src.res[0]
            ),
            "yres": abs(
                src.res[1]
            ),
            "width": src.width,
            "height": src.height,
            "crs": str(
                src.crs
            ),
        })


qa_rows.append({
    "dataset": "FINAL_ML_OUTPUT",
    "xres": abs(
        TARGET_PROFILE[
            "transform"
        ].a
    ),
    "yres": abs(
        TARGET_PROFILE[
            "transform"
        ].e
    ),
    "width": TARGET_WIDTH,
    "height": TARGET_HEIGHT,
    "crs": str(
        TARGET_PROFILE[
            "crs"
        ]
    ),
})


resolution_qa = pd.DataFrame(
    qa_rows
)

display(
    resolution_qa
)

resolution_qa.to_csv(
    QA_DIR
    / "resolution_QA.csv",
    index=False
)


sample = next(
    iter(
        generated.values()
    ),
    None
)

if sample:

    with rasterio.open(
        sample
    ) as src:

        a = src.read(
            1
        )

        outside_ok = np.all(
            a[
                ~TARGET_MASK
            ]
            == src.nodata
        )

        inside_valid = (
            np.isfinite(
                a[
                    TARGET_MASK
                ]
            )
            &
            (
                a[
                    TARGET_MASK
                ]
                != src.nodata
            )
        )

        print(
            "Sample output:",
            sample
        )

        print(
            "Resolution:",
            src.res
        )

        print(
            "Outside Khulna = NoData:",
            outside_ok
        )

        print(
            "Inside Khulna coverage:",
            (
                100
                * inside_valid.sum()
                / TARGET_PIXELS
            )
        )


In [ ]:
# ============================================================
# 14. ANNUAL MAPS + FINAL 4 × 3 PAPER-STYLE FIGURE
# ============================================================

def annual_from_generated(
    combo,
    model
):
    paths = [
        generated.get(
            (
                combo,
                model,
                m
            )
        )
        for m in range(
            1,
            13
        )
    ]

    missing = [
        m + 1
        for m, p in enumerate(
            paths
        )
        if (
            p is None
            or not Path(
                p
            ).exists()
        )
    ]

    if missing:
        return (
            None,
            missing
        )

    stack = []

    for p in paths:

        with rasterio.open(
            p
        ) as src:

            a = src.read(
                1
            ).astype(
                "float32"
            )

            if src.nodata is not None:
                a[
                    np.isclose(
                        a,
                        src.nodata
                    )
                ] = np.nan

            stack.append(
                a
            )


    stack = np.stack(
        stack
    )

    valid = (
        np.all(
            np.isfinite(
                stack
            ),
            axis=0
        )
        & TARGET_MASK
    )

    annual = np.full(
        TARGET_SHAPE,
        np.nan,
        dtype="float32"
    )

    annual[
        valid
    ] = np.sum(
        stack[
            :,
            valid
        ],
        axis=0
    )

    return (
        annual,
        []
    )


annual = {}
annual_rows = []


for model in MAP_MODELS:

    for combo in MAP_COMBOS:

        if (
            combo,
            model
        ) not in trained:
            continue

        a, missing = annual_from_generated(
            combo,
            model
        )

        if a is None:

            annual_rows.append({
                "combo": combo,
                "model": model,
                "status": "SKIP",
                "missing_months": str(
                    missing
                ),
                "coverage_pct_inside_khulna": 0.0,
            })

            continue


        annual[
            (
                combo,
                model
            )
        ] = a

        out_tif = (
            ANNUAL_DIR
            / (
                f"annual_{TEST_YEAR}_{combo}_"
                f"{model.replace(' ', '_')}.tif"
            )
        )

        write_masked_tif(
            out_tif,
            a
        )

        valid = (
            np.isfinite(
                a
            )
            & TARGET_MASK
        )

        vals = a[
            valid
        ]

        annual_rows.append({
            "combo": combo,
            "model": model,
            "status": "OK",
            "missing_months": "",
            "coverage_pct_inside_khulna": (
                100.0
                * valid.sum()
                / TARGET_PIXELS
            ),
            "min_mm_year": (
                float(
                    np.min(
                        vals
                    )
                )
                if vals.size
                else np.nan
            ),
            "p05_mm_year": (
                float(
                    np.percentile(
                        vals,
                        5
                    )
                )
                if vals.size
                else np.nan
            ),
            "median_mm_year": (
                float(
                    np.median(
                        vals
                    )
                )
                if vals.size
                else np.nan
            ),
            "mean_mm_year": (
                float(
                    np.mean(
                        vals
                    )
                )
                if vals.size
                else np.nan
            ),
            "p95_mm_year": (
                float(
                    np.percentile(
                        vals,
                        95
                    )
                )
                if vals.size
                else np.nan
            ),
            "max_mm_year": (
                float(
                    np.max(
                        vals
                    )
                )
                if vals.size
                else np.nan
            ),
            "annual_tif": str(
                out_tif
            ),
        })


annual_qa = pd.DataFrame(
    annual_rows
)

display(
    annual_qa
)

annual_qa.to_csv(
    QA_DIR
    / "annual_generation_QA.csv",
    index=False
)


vals = []

for a in annual.values():

    v = a[
        np.isfinite(
            a
        )
    ]

    if v.size:
        vals.append(
            v
        )


if not vals:
    raise RuntimeError(
        "No annual map was generated."
    )


vals = np.concatenate(
    vals
)

vmin, vmax = np.nanpercentile(
    vals,
    [
        1,
        99
    ]
)


extent = [
    TARGET_BOUNDS.left,
    TARGET_BOUNDS.right,
    TARGET_BOUNDS.bottom,
    TARGET_BOUNDS.top,
]


fig, axes = plt.subplots(
    len(
        MAP_MODELS
    ),
    len(
        MAP_COMBOS
    ),
    figsize=(
        3.2
        * len(
            MAP_COMBOS
        ),
        3.0
        * len(
            MAP_MODELS
        )
    ),
    squeeze=False,
    sharex=True,
    sharey=True
)


im = None


for i, model in enumerate(
    MAP_MODELS
):

    for j, combo in enumerate(
        MAP_COMBOS
    ):

        ax = axes[
            i,
            j
        ]

        a = annual.get(
            (
                combo,
                model
            )
        )

        if a is None:

            ax.axis(
                "off"
            )

            ax.text(
                0.5,
                0.5,
                "Not generated",
                ha="center",
                va="center",
                transform=ax.transAxes
            )

            continue


        im = ax.imshow(
            a,
            extent=extent,
            origin="upper",
            cmap="turbo",
            vmin=vmin,
            vmax=vmax,
            interpolation="nearest"
        )


        boundary.boundary.plot(
            ax=ax,
            color="black",
            linewidth=0.7
        )


        ax.set_aspect(
            "equal"
        )


        if i == 0:

            ax.set_title(
                combo,
                fontweight="bold",
                fontsize=12
            )


        if j == 0:

            ax.set_ylabel(
                model
                + "\nLatitude"
            )

        else:

            ax.set_yticklabels(
                []
            )


        if i == len(
            MAP_MODELS
        ) - 1:

            ax.set_xlabel(
                "Longitude"
            )

        else:

            ax.set_xticklabels(
                []
            )


cb = fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    shrink=0.78,
    pad=0.02
)

cb.set_label(
    f"Total annual rainfall (mm), {TEST_YEAR}"
)


fig.suptitle(
    f"Total annual downscaled rainfall over Khulna District ({TEST_YEAR})\n"
    "Masked ML output grid: 0.005 degree (~500–550 m)",
    y=0.995,
    fontweight="bold",
    fontsize=15
)


final_png = (
    FIG_DIR
    / "FINAL_Annual_Rainfall_Paper_Style_FULL_COVER.png"
)

final_pdf = (
    FIG_DIR
    / "FINAL_Annual_Rainfall_Paper_Style_FULL_COVER.pdf"
)


fig.savefig(
    final_png,
    dpi=FIG_DPI,
    bbox_inches="tight"
)

fig.savefig(
    final_pdf,
    bbox_inches="tight"
)

plt.show()


print(
    "FINAL annual figure:",
    final_png
)


In [ ]:
# ============================================================
# 15. OBSERVED 2022 ANNUAL GAUGE RAINFALL
# ============================================================

observed_annual = (
    data[
        data["year"] == TEST_YEAR
    ]
    .groupby(
        "station_id",
        as_index=False
    )
    .agg(
        observed_annual_mm=(
            TARGET,
            "sum"
        ),
        months=(
            TARGET,
            "count"
        ),
    )
)

display(
    observed_annual
)

observed_annual.to_csv(
    TABLE_DIR
    / "Observed_Annual_Gauge_Rainfall_2022.csv",
    index=False
)


fig, ax = plt.subplots(
    figsize=(
        8,
        5
    )
)

ax.bar(
    observed_annual[
        "station_id"
    ],
    observed_annual[
        "observed_annual_mm"
    ]
)

ax.set_xlabel(
    "Station"
)

ax.set_ylabel(
    "Observed annual rainfall (mm/year)"
)

ax.set_title(
    "Observed Annual Rainfall at Gauge Stations — 2022"
)

fig.tight_layout()

fig.savefig(
    FIG_DIR
    / "Observed_Annual_Gauge_Rainfall_2022.png",
    dpi=FIG_DPI,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# 16. OUTPUT MANIFEST
# ============================================================

manifest = []

for folder in [
    TABLE_DIR,
    FIG_DIR,
    MAP_DIR,
    ANNUAL_DIR,
    QA_DIR,
    MODEL_DIR,
]:

    if folder.exists():

        for p in sorted(
            folder.rglob(
                "*"
            )
        ):

            if p.is_file():

                try:
                    rel = p.relative_to(
                        PROJECT_ROOT
                    )
                except Exception:
                    rel = p

                manifest.append(
                    str(
                        rel
                    )
                )


manifest_df = pd.DataFrame({
    "output": manifest
})

manifest_df.to_csv(
    OUT_ROOT
    / "output_manifest.csv",
    index=False
)

display(
    manifest_df
)

print(
    "\nDONE:",
    OUT_ROOT
)

print(
    "\nIMPORTANT OUTPUTS:"
)

print(
    " -",
    TABLE_DIR
    / "Table_1_paper_style.png"
)

print(
    " -",
    FIG_DIR
    / "Fig_3_spatial_MAE.png"
)

print(
    " -",
    FIG_DIR
    / "Fig_4_spatial_RMSE.png"
)

print(
    " -",
    FIG_DIR
    / "Fig_5_spatial_PCC.png"
)

print(
    " -",
    FIG_DIR
    / "FINAL_Annual_Rainfall_Paper_Style_FULL_COVER.png"
)

print(
    " -",
    QA_DIR
    / "map_generation_QA.csv"
)

print(
    " -",
    QA_DIR
    / "full_cover_fill_QA.csv"
)

print(
    " -",
    QA_DIR
    / "annual_generation_QA.csv"
)

print(
    " -",
    QA_DIR
    / "resolution_QA.csv"
)
